In [ ]:
import torch
import matplotlib.pyplot as plt

from metrics import align_channels, abs_error_map


CHANNEL_NAMES = {
    0: "K",
    1: "P",
    2: "phi",
}


def unpack_dataset_item(item):
    if len(item) == 3:
        feat, label, mask = item
    else:
        feat, label = item
        mask = None

    return feat, label, mask


def predict_one(model, feat, device):
    model.eval()

    with torch.no_grad():
        pred = model(feat.unsqueeze(0).to(device))
        pred = pred.cpu().squeeze(0)

    return pred


def compare_models_on_item(
    models,
    dataset,
    idx,
    device,
    channel=0,
    show_error=True,
    save_path=None,
):
    """
    Visual comparison for one dataset item.

    Args:
        models:
            dict like {"model name": model}
            OR a single model

        dataset:
            validation/test dataset

        idx:
            dataset index / timestep-like index

        device:
            DEVICE

        channel:
            0 = K, 1 = P, 2 = phi

        show_error:
            if True, show prediction and absolute error for each model
            if False, only show predictions

    Example:
        compare_models_on_item(
            models={"SplitNet Darcy": model1, "UNet": model2},
            dataset=val_loader.dataset,
            idx=20,
            device=DEVICE,
            channel=0
        )
    """

    # Allow single model input
    if not isinstance(models, dict):
        models = {"Model": models}

    item = dataset[idx]
    feat, label, mask = unpack_dataset_item(item)

    predictions = {}

    for name, model in models.items():
        pred = predict_one(model, feat, device)
        predictions[name] = pred

    # Align label to first prediction channel count
    first_pred = next(iter(predictions.values()))
    label_b = align_channels(label.unsqueeze(0), first_pred.unsqueeze(0))
    label = label_b.squeeze(0)

    channel_name = CHANNEL_NAMES.get(channel, f"channel {channel}")

    n_models = len(models)

    if show_error:
        n_cols = 2 + (2 * n_models)
    else:
        n_cols = 2 + n_models

    fig, axes = plt.subplots(
        1,
        n_cols,
        figsize=(4 * n_cols, 4),
        squeeze=False,
    )

    axes = axes[0]
    col = 0

    # Input/masked feature
    im = axes[col].imshow(feat[channel].numpy())
    axes[col].set_title(f"Input {channel_name}")
    axes[col].axis("off")
    plt.colorbar(im, ax=axes[col], fraction=0.046)
    col += 1

    # Ground truth
    im = axes[col].imshow(label[channel].numpy())
    axes[col].set_title(f"Ground Truth {channel_name}")
    axes[col].axis("off")
    plt.colorbar(im, ax=axes[col], fraction=0.046)
    col += 1

    # Model predictions
    for name, pred in predictions.items():
        im = axes[col].imshow(pred[channel].numpy())
        axes[col].set_title(f"{name}\nPrediction")
        axes[col].axis("off")
        plt.colorbar(im, ax=axes[col], fraction=0.046)
        col += 1

        if show_error:
            error = torch.abs(pred[channel] - label[channel])

            im = axes[col].imshow(error.numpy())
            axes[col].set_title(f"{name}\nAbs Error")
            axes[col].axis("off")
            plt.colorbar(im, ax=axes[col], fraction=0.046)
            col += 1

    fig.suptitle(f"Model Comparison on idx={idx}, channel={channel_name}", fontsize=16)
    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, bbox_inches="tight", dpi=200)

    plt.show()

    return predictions

In [ ]:
models = {
    "SplitNet + Darcy": splitnet_model,
    "Attention UNet": attn_unet_model,
    "UNetFixed": prof_unet_model,
}

preds = compare_models_on_item(
    models=models,
    dataset=val_loader.dataset,
    idx=50,
    device=DEVICE,
    channel=0,   # K
    show_error=True,
    save_path="comparison_K_idx50.png",
)